# HydroTwin Digital Twin Development

This notebook develops a historical data-driven digital twin prototype for two hydropower components:

1. Turbine Guide Bearing (TGB)
2. Upper Guide Bearing (UGB)

Both components belong to the same generating unit (G1) and share operating information, including unit speed and generator power.

The digital twin integrates historical sensor records, machine learning vibration forecasts, and synchronized component states.

The prototype uses November 2020 observations for historical replay. It does not claim a live physical plant connection.

The three trained forecasting models are Linear Regression, LSTM, and GRU.

## 1. Import Libraries

The libraries required for loading historical sensor observations, prediction records, and digital twin data structures are imported.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

## 2. Project Directories

The digital twin uses the cleaned hydropower dataset and previously generated November prediction records.

All paths are defined relative to the HydroTwin project directory.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"

RESULTS_DIR = PROJECT_ROOT / "results"

DIGITAL_TWIN_DIR = PROJECT_ROOT / "digital_twin"

DIGITAL_TWIN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project Root:", PROJECT_ROOT)

print(
    "Digital Twin Directory:",
    DIGITAL_TWIN_DIR
)

## 3. Load Historical Sensor and Forecast Data

The cleaned hydropower sensor dataset and the saved model predictions are loaded.

The sensor records provide the historical operating state of the generating unit and its two bearing components.

The prediction records provide timestamp-aligned one-step-ahead vibration forecasts generated by the finalized machine learning models.

In [ ]:
# Load cleaned hydropower dataset

sensor_data = pd.read_csv(
    PREPARED_DIR / "hydrotwin_cleaned_data.csv"
)

# Load TGB model predictions

tgb_predictions = pd.read_csv(
    RESULTS_DIR / "tgb_final_predictions.csv"
)

# Load UGB model predictions

ugb_predictions = pd.read_csv(
    RESULTS_DIR / "ugb_final_predictions.csv"
)

print("Cleaned sensor records:", sensor_data.shape)

print("TGB prediction records:", tgb_predictions.shape)

print("UGB prediction records:", ugb_predictions.shape)

## 4. Inspect Historical Data Structure

The dataset columns and sample records are inspected before constructing synchronized digital twin states.

This ensures that the correct sensor variables and timestamps are used for each bearing component.

In [ ]:
print("CLEANED SENSOR DATA COLUMNS:")
print(sensor_data.columns.tolist())

print("\nTGB PREDICTION COLUMNS:")
print(tgb_predictions.columns.tolist())

print("\nUGB PREDICTION COLUMNS:")
print(ugb_predictions.columns.tolist())

print("\nSENSOR DATA SAMPLE:")
display(sensor_data.head(3))

print("\nTGB PREDICTION SAMPLE:")
display(tgb_predictions.head(3))

## 5. Integrated Generating Unit Digital Twin

HydroTwin represents one hydropower generating unit (G1) containing two monitored bearing components:

1. Turbine Guide Bearing (TGB)
2. Upper Guide Bearing (UGB)

Both bearings share generator power and unit speed measurements while maintaining individual vibration, temperature, and vibration prediction values.

The system will use one integrated 3D model to visualize both bearing components.

Historical sensor measurements and machine learning predictions will be synchronized using timestamps to represent the generating unit's state.

## 6. Timestamp Preparation

The timestamps in the cleaned sensor dataset and the two prediction datasets are converted into datetime format.

This ensures that the sensor measurements and model predictions can be matched correctly during synchronization.

In [ ]:
sensor_data["timestamp"] = pd.to_datetime(
    sensor_data["timestamp"]
)

tgb_predictions["timestamp"] = pd.to_datetime(
    tgb_predictions["timestamp"]
)

ugb_predictions["timestamp"] = pd.to_datetime(
    ugb_predictions["timestamp"]
)

print("Sensor Data:")
print("Start:", sensor_data["timestamp"].min())
print("End:", sensor_data["timestamp"].max())

print("\nTGB Predictions:")
print("Start:", tgb_predictions["timestamp"].min())
print("End:", tgb_predictions["timestamp"].max())

print("\nUGB Predictions:")
print("Start:", ugb_predictions["timestamp"].min())
print("End:", ugb_predictions["timestamp"].max())

## 7. Verify Component Timestamp Alignment

The TGB and UGB prediction timestamps are checked to confirm that both bearing components can be represented in one synchronized digital twin.

Timestamp uniqueness is also verified to prevent duplicate digital twin states.

In [ ]:
timestamps_aligned = np.array_equal(
    tgb_predictions["timestamp"].to_numpy(),
    ugb_predictions["timestamp"].to_numpy()
)

print(
    "TGB and UGB timestamps aligned:",
    timestamps_aligned
)

print(
    "TGB prediction records:",
    len(tgb_predictions)
)

print(
    "UGB prediction records:",
    len(ugb_predictions)
)

print(
    "Unique sensor timestamps:",
    sensor_data["timestamp"].is_unique
)

print(
    "Unique TGB timestamps:",
    tgb_predictions["timestamp"].is_unique
)

print(
    "Unique UGB timestamps:",
    ugb_predictions["timestamp"].is_unique
)

## 8. Construct Integrated Digital Twin Dataset

The TGB and UGB prediction records are merged with the cleaned sensor dataset using timestamps.

Each row represents the historical generating unit state for both bearings at one timestamp.

The model predictions represent one-step-ahead forecasts generated from the preceding 36 historical sensor observations.

The resulting dataset provides a common data source for the integrated HydroTwin digital twin and 3D visualization.

In [ ]:
# Rename TGB prediction columns

tgb_forecasts = tgb_predictions[
    [
        "timestamp",
        "linear_regression_prediction",
        "lstm_prediction",
        "gru_prediction"
    ]
].rename(columns={
    "linear_regression_prediction": "tgb_lr_prediction",
    "lstm_prediction": "tgb_lstm_prediction",
    "gru_prediction": "tgb_gru_prediction"
})

# Rename UGB prediction columns

ugb_forecasts = ugb_predictions[
    [
        "timestamp",
        "linear_regression_prediction",
        "lstm_prediction",
        "gru_prediction"
    ]
].rename(columns={
    "linear_regression_prediction": "ugb_lr_prediction",
    "lstm_prediction": "ugb_lstm_prediction",
    "gru_prediction": "ugb_gru_prediction"
})

# Merge into one integrated dataset

digital_twin_data = (
    tgb_forecasts
    .merge(
        ugb_forecasts,
        on="timestamp",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        sensor_data,
        on="timestamp",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print(
    "Integrated digital twin records:",
    len(digital_twin_data)
)

print(
    "Integrated dataset shape:",
    digital_twin_data.shape
)

## 9. Integrated Dataset Verification

The synchronized digital twin dataset is verified to confirm that all required sensor measurements and forecasting outputs are present.

Missing values and timestamp alignment are checked before developing the digital twin state management system.

In [ ]:
required_columns = [
    "timestamp",
    "generator_power",
    "unit_speed",

    "tgb_vibration",
    "tgb_temperature",

    "ugb_vibration",
    "ugb_temperature",

    "tgb_lr_prediction",
    "tgb_lstm_prediction",
    "tgb_gru_prediction",

    "ugb_lr_prediction",
    "ugb_lstm_prediction",
    "ugb_gru_prediction"
]

missing_columns = [
    col for col in required_columns
    if col not in digital_twin_data.columns
]

print("Missing required columns:", missing_columns)

if not missing_columns:

    print(
        "Total missing values:",
        digital_twin_data[
            required_columns
        ].isna().sum().sum()
    )

    print(
        "Unique timestamps:",
        digital_twin_data["timestamp"].is_unique
    )

    display(
        digital_twin_data[
            required_columns
        ].head(5)
    )

## 10. Verify Actual Vibration Alignment

The actual TGB and UGB vibration measurements in the integrated dataset are compared against the original prediction records.

This verification ensures that sensor measurements and forecasting targets correspond to the same historical timestamps.

In [ ]:
# Verify TGB actual readings

tgb_actual_check = (
    digital_twin_data[["timestamp", "tgb_vibration"]]
    .merge(
        tgb_predictions[["timestamp", "actual_vibration"]],
        on="timestamp",
        validate="one_to_one"
    )
)

tgb_match = np.allclose(
    tgb_actual_check["tgb_vibration"],
    tgb_actual_check["actual_vibration"]
)

# Verify UGB actual readings

ugb_actual_check = (
    digital_twin_data[["timestamp", "ugb_vibration"]]
    .merge(
        ugb_predictions[["timestamp", "actual_vibration"]],
        on="timestamp",
        validate="one_to_one"
    )
)

ugb_match = np.allclose(
    ugb_actual_check["ugb_vibration"],
    ugb_actual_check["actual_vibration"]
)

print("TGB actual vibration aligned:", tgb_match)
print("UGB actual vibration aligned:", ugb_match)

assert tgb_match and ugb_match

## 11. Save Integrated Digital Twin Dataset

The synchronized generating-unit dataset is saved for use in the digital twin state management system and Streamlit dashboard.

The dataset contains historical measurements and timestamp-aligned forecasts for both TGB and UGB.

In [ ]:
digital_twin_path = (
    DIGITAL_TWIN_DIR / "g1_integrated_replay.csv"
)

digital_twin_data.to_csv(
    digital_twin_path,
    index=False
)

print("Dataset saved:", digital_twin_path.exists())
print("Total records:", len(digital_twin_data))

## 12. HydroTwin Generating Unit State Management

A Python class is developed to represent the integrated historical digital twin of generating unit G1.

The class manages synchronized bearing states, shared operating measurements, historical replay progression, and next-step vibration forecasts.

TGB and UGB are represented as monitored components within the same generating unit.

The initial model selections are GRU for TGB and LSTM for UGB, based on September validation results.

Forecasts are displayed for the next available 10-minute timestamp only when the corresponding observations are continuous.

In [ ]:
%%writefile ../digital_twin/hydrotwin_core.py

import pandas as pd


class HydroTwinGeneratingUnit:

    MODEL_COLUMNS = {
        "TGB": {
            "Linear Regression": "tgb_lr_prediction",
            "LSTM": "tgb_lstm_prediction",
            "GRU": "tgb_gru_prediction"
        },

        "UGB": {
            "Linear Regression": "ugb_lr_prediction",
            "LSTM": "ugb_lstm_prediction",
            "GRU": "ugb_gru_prediction"
        }
    }

    def __init__(self, data):

        self.data = (
            data.copy()
            .sort_values("timestamp")
            .reset_index(drop=True)
        )

        self.data["timestamp"] = pd.to_datetime(
            self.data["timestamp"]
        )

        if self.data.empty:
            raise ValueError("Digital twin dataset is empty.")

        if self.data["timestamp"].duplicated().any():
            raise ValueError("Duplicate timestamps detected.")

        self.position = 0

        # Models selected using September validation
        self.selected_models = {
            "TGB": "GRU",
            "UGB": "LSTM"
        }

    def get_bearing_state(self, component, row, next_row):

        prefix = component.lower()

        # Next-timestamp model predictions
        if next_row is not None:

            predictions = {
                model: float(next_row[column])
                for model, column in
                self.MODEL_COLUMNS[component].items()
            }

        else:

            predictions = {
                model: None
                for model in self.MODEL_COLUMNS[component]
            }

        selected_model = self.selected_models[component]

        return {
            "component": component,

            "vibration": float(
                row[f"{prefix}_vibration"]
            ),

            "temperature": float(
                row[f"{prefix}_temperature"]
            ),

            "selected_model": selected_model,

            "next_prediction": predictions[selected_model],

            "all_predictions": predictions
        }

    def get_state(self):

        row = self.data.iloc[self.position]

        current_time = row["timestamp"]

        next_row = None
        forecast_time = None

        if self.position + 1 < len(self.data):

            candidate = self.data.iloc[self.position + 1]

            expected_time = (
                current_time + pd.Timedelta(minutes=10)
            )

            if candidate["timestamp"] == expected_time:

                next_row = candidate
                forecast_time = expected_time

        return {
            "unit_id": "G1",

            "timestamp": current_time,

            "forecast_timestamp": forecast_time,

            "replay_position": self.position,

            "total_records": len(self.data),

            "generator_power": float(
                row["generator_power"]
            ),

            "unit_speed": float(
                row["unit_speed"]
            ),

            "TGB": self.get_bearing_state(
                "TGB", row, next_row
            ),

            "UGB": self.get_bearing_state(
                "UGB", row, next_row
            )
        }

    def next_step(self):

        if self.position + 1 >= len(self.data):
            return None

        self.position += 1

        return self.get_state()

    def reset(self):

        self.position = 0

        return self.get_state()

    def select_model(self, component, model):

        component = component.upper()

        if component not in self.MODEL_COLUMNS:
            raise ValueError("Unknown component.")

        if model not in self.MODEL_COLUMNS[component]:
            raise ValueError("Unknown model.")

        self.selected_models[component] = model

## 13. Initialize and Test the Integrated Digital Twin

The HydroTwin generating unit is initialized using the synchronized historical dataset.

The initial state is examined to verify that both TGB and UGB are updated at the same timestamp and that valid next-step forecasts are available.

In [ ]:
import sys

sys.path.insert(
    0,
    str(DIGITAL_TWIN_DIR)
)

from hydrotwin_core import HydroTwinGeneratingUnit

# Initialize the integrated twin

hydrotwin = HydroTwinGeneratingUnit(
    digital_twin_data
)

# Retrieve initial state

state = hydrotwin.get_state()

print("Generating Unit:", state["unit_id"])

print("Current Timestamp:", state["timestamp"])

print(
    "Forecast Timestamp:",
    state["forecast_timestamp"]
)

print("\nShared Operating Conditions")

print("Unit Speed:", state["unit_speed"])
print("Generator Power:", state["generator_power"])

print("\nTGB State")
print(state["TGB"])

print("\nUGB State")
print(state["UGB"])

## 14. Historical Replay Verification

The integrated digital twin advances through historical observations at 10-minute recorded intervals.

Both bearing states are updated together, while model forecasts remain aligned with their intended future target timestamps.

This verifies the historical synchronization mechanism that will later support interactive playback in Streamlit.

In [ ]:
# Reset the twin to its first historical state

hydrotwin.reset()

# Replay the first five states

for i in range(5):

    state = hydrotwin.get_state()

    print(
        f"Step {i + 1} | "
        f"Time: {state['timestamp']} | "
        f"TGB: {state['TGB']['vibration']} | "
        f"UGB: {state['UGB']['vibration']}"
    )

    hydrotwin.next_step()

## 15. Component State Viewer

A component state viewer is developed to display sensor measurements and vibration forecasts for either TGB or UGB.

Both bearings belong to one integrated HydroTwin generating unit.

The viewer extracts the selected bearing's measurements from the synchronized generating-unit state and will later support component selection within the interactive 3D model.

In [ ]:
def get_component_view(unit_state, component):

    component = component.upper()

    if component not in ["TGB", "UGB"]:
        raise ValueError("Component must be TGB or UGB.")

    bearing = unit_state[component]

    return {
        "Generating Unit": unit_state["unit_id"],

        "Component": component,

        "Current Timestamp": unit_state["timestamp"],

        "Forecast Timestamp": unit_state["forecast_timestamp"],

        "Actual Vibration": bearing["vibration"],

        "Temperature": bearing["temperature"],

        "Selected Model": bearing["selected_model"],

        "Predicted Vibration": bearing["next_prediction"],

        "Unit Speed": unit_state["unit_speed"],

        "Generator Power": unit_state["generator_power"]
    }


print("Component viewer created successfully.")

## 16. Verify TGB and UGB Component Views

The component viewer is tested using the integrated digital twin.

Both bearing views must display the same generating-unit timestamp and shared operating measurements, while showing their respective vibration, temperature, and forecasting values.

In [ ]:
# Reset to the first historical record

hydrotwin.reset()

# Obtain the synchronized generating-unit state

unit_state = hydrotwin.get_state()

# Create both component views

tgb_view = get_component_view(
    unit_state,
    "TGB"
)

ugb_view = get_component_view(
    unit_state,
    "UGB"
)

print("TGB COMPONENT VIEW")
display(pd.DataFrame([tgb_view]).T)

print("\nUGB COMPONENT VIEW")
display(pd.DataFrame([ugb_view]).T)

## 17. Forecasting Model Selection

The integrated digital twin supports switching between the three previously trained forecasting models for each monitored bearing.

The forecasting model can be changed independently for TGB and UGB while retaining the same historical sensor state.

The default selections remain GRU for TGB and LSTM for UGB, based on September validation performance.

In [ ]:
# Temporarily change the model for both components

hydrotwin.select_model(
    "TGB",
    "LSTM"
)

hydrotwin.select_model(
    "UGB",
    "GRU"
)

# Retrieve the updated unit state

updated_state = hydrotwin.get_state()

model_switch_results = pd.DataFrame([
    get_component_view(updated_state, "TGB"),
    get_component_view(updated_state, "UGB")
])

display(
    model_switch_results[
        [
            "Component",
            "Selected Model",
            "Actual Vibration",
            "Predicted Vibration",
            "Forecast Timestamp"
        ]
    ]
)

In [ ]:
hydrotwin.select_model("TGB", "GRU")
hydrotwin.select_model("UGB", "LSTM")

print("TGB selected model:",
      hydrotwin.get_state()["TGB"]["selected_model"])

print("UGB selected model:",
      hydrotwin.get_state()["UGB"]["selected_model"])

## 18. Integrated 3D Model Component Mapping

A conceptual 3D component mapping is defined for the HydroTwin generating-unit visualization.

The system uses one integrated 3D representation with two individually identifiable bearing components.

The Upper Guide Bearing (UGB) is represented in the upper generator-shaft region, while the Turbine Guide Bearing (TGB) is represented near the turbine-shaft region.

The mapping identifies which digital twin component state should be displayed when a bearing is selected.

The planned geometry is conceptual and does not represent verified dimensions of the actual hydropower plant.

In [ ]:
component_3d_mapping = {
    "G1": {
        "name": "Hydropower Generating Unit G1",
        "components": {
            "UGB": {
                "label": "Upper Guide Bearing",
                "position": "Upper generator shaft",
                "state_key": "UGB",
                "color": "#F59E0B"
            },
            "TGB": {
                "label": "Turbine Guide Bearing",
                "position": "Lower turbine shaft",
                "state_key": "TGB",
                "color": "#3B82F6"
            }
        }
    }
}

print("Integrated 3D component mapping:")

for component, config in (
    component_3d_mapping["G1"]["components"].items()
):
    print(
        f"{component}: "
        f"{config['label']} | "
        f"{config['position']}"
    )

In [ ]:
import json

mapping_path = (
    DIGITAL_TWIN_DIR / "component_3d_mapping.json"
)

with open(mapping_path, "w", encoding="utf-8") as file:
    json.dump(
        component_3d_mapping,
        file,
        indent=4
    )

print("3D mapping saved:", mapping_path.exists())

## 19. Integrated CAD Assembly Visualization

A conceptual CAD-style vertical hydropower generating unit was developed to represent the HydroTwin digital twin.

The assembly includes the generator, rotor, stator, vertical shaft, turbine runner, Turbine Guide Bearing (TGB), and Upper Guide Bearing (UGB).

The TGB and UGB components are individually identifiable within a single integrated assembly.

The CAD geometry is an engineering-inspired conceptual representation and does not reproduce verified dimensions of the actual hydropower plant.

In [ ]:
import json

CAD_DIR = DIGITAL_TWIN_DIR / "assets"

GLB_PATH = CAD_DIR / "HydroTwin_G1_Interactive.glb"

STEP_PATH = CAD_DIR / "HydroTwin_G1_Conceptual_Assembly.step"

MANIFEST_PATH = CAD_DIR / "HydroTwin_G1_manifest.json"

for path in [GLB_PATH, STEP_PATH, MANIFEST_PATH]:
    print(path.name, ":", path.exists())

assert all(
    path.exists()
    for path in [GLB_PATH, STEP_PATH, MANIFEST_PATH]
)

with open(MANIFEST_PATH, encoding="utf-8") as file:
    cad_manifest = json.load(file)

print("Total named parts:", cad_manifest["part_count"])
print("Monitored components:", list(cad_manifest["bearing_components"]))

## 20. Interactive CAD Model Inspection

The integrated GLB assembly is displayed in an interactive Plotly viewer.

The model can be rotated and zoomed, and both monitored bearing components are visually highlighted.

This visualization forms the basis for future integration with the HydroTwin Streamlit dashboard.

In [ ]:
%pip install trimesh plotly

In [ ]:
import sys

sys.path.insert(0, str(DIGITAL_TWIN_DIR))

from hydrotwin_cad_viewer import cad_preview

fig = cad_preview(
    GLB_PATH,
    selected_component="TGB"
)

fig.show()

## 22. Improved CAD Assembly Visualization

The integrated HydroTwin CAD visualization is enhanced with improved camera positioning, transparent outer housing, and bearing component highlighting.

The generating unit consists of one conceptual vertical-axis assembly containing TGB and UGB.

The visualization allows interactive rotation, zooming, and inspection of internal mechanical components.

In [ ]:
import sys
import importlib

sys.path.insert(0, str(DIGITAL_TWIN_DIR))

import hydrotwin_cad_viewer

# Reload the updated viewer without restarting the kernel
importlib.reload(hydrotwin_cad_viewer)

from hydrotwin_cad_viewer import cad_preview

fig = cad_preview(
    GLB_PATH,
    selected_component="TGB",
    housing_opacity=0.24
)

fig.show()

In [ ]:
fig_ugb = cad_preview(
    GLB_PATH,
    selected_component="UGB",
    housing_opacity=0.24
)

fig_ugb.show()

## 23. Connect 3D Assembly with Digital Twin State

The integrated CAD-style generating-unit assembly is connected to the HydroTwin historical state management system.

Selecting TGB or UGB highlights the corresponding bearing in the 3D model and displays its historical vibration, temperature, and predicted vibration.

Both components share the same generating-unit timestamp, unit speed, and generator power.

The forecasts are generated from historical input windows and represent the next 10-minute vibration reading.

In [ ]:
def show_hydrotwin_component(component="TGB"):

    component = component.upper()

    if component not in ["TGB", "UGB"]:
        raise ValueError("Select TGB or UGB.")

    # Get current historical digital twin state
    unit_state = hydrotwin.get_state()

    # Retrieve the selected component's information
    component_view = get_component_view(
        unit_state,
        component
    )

    # Display component information
    print("HYDROTWIN GENERATING UNIT G1")
    print("Selected Component:", component)

    display(
        pd.DataFrame(
            [component_view]
        ).T
    )

    # Highlight the selected bearing
    fig = cad_preview(
        GLB_PATH,
        selected_component=component,
        housing_opacity=0.24
    )

    fig.show()

## 24. TGB CAD and Sensor Integration Test

The TGB component is selected in the integrated generating-unit assembly.

The visualization highlights TGB and displays the synchronized historical sensor values and next-step vibration forecast.

In [ ]:
# Reset to the first historical observation
hydrotwin.reset()

# Display TGB
show_hydrotwin_component("TGB")

## 25. UGB CAD and Sensor Integration Test

The UGB component is selected within the same integrated generating-unit assembly.

The visualization highlights UGB and displays its corresponding historical sensor measurements and next-step vibration prediction.

In [ ]:
show_hydrotwin_component("UGB")

## 26. Integrated 3D Historical Replay Test

The generating-unit digital twin advances to the next historical observation.

The updated component measurements and vibration forecasts are displayed alongside the integrated CAD assembly.

This demonstrates synchronization between the digital twin state manager and the 3D visualization.

In [ ]:
# Advance to next historical record
next_state = hydrotwin.next_step()

print("New historical timestamp:", next_state["timestamp"])
print("Next forecast timestamp:", next_state["forecast_timestamp"])

# Show updated TGB state
show_hydrotwin_component("TGB")

## 27. Digital Twin Deployment Readiness

The integrated HydroTwin data, state management module, CAD viewer, and 3D model assets are verified before dashboard development.

The digital twin uses a single conceptual generating-unit assembly with independently monitored TGB and UGB components.

The synchronized historical dataset and saved forecasting results will be used for Streamlit integration.

In [ ]:
required_files = {
    "Integrated Replay Dataset":
        DIGITAL_TWIN_DIR / "g1_integrated_replay.csv",

    "Digital Twin Core":
        DIGITAL_TWIN_DIR / "hydrotwin_core.py",

    "CAD Viewer":
        DIGITAL_TWIN_DIR / "hydrotwin_cad_viewer.py",

    "3D Component Mapping":
        DIGITAL_TWIN_DIR / "component_3d_mapping.json",

    "Interactive CAD Model":
        DIGITAL_TWIN_DIR / "assets" /
        "HydroTwin_G1_Interactive.glb",

    "CAD Manifest":
        DIGITAL_TWIN_DIR / "assets" /
        "HydroTwin_G1_manifest.json"
}

for name, path in required_files.items():
    print(f"{name}: {path.exists()}")

assert all(
    path.exists()
    for path in required_files.values()
)

print("\nHydroTwin is ready for dashboard development.")

## 28. Conclusion

An integrated historical data-driven digital twin prototype was developed for hydropower generating unit G1.

The system represents two monitored bearing components: the Turbine Guide Bearing (TGB) and Upper Guide Bearing (UGB).

A conceptual CAD-style vertical hydropower assembly was integrated with synchronized historical sensor measurements and machine learning vibration forecasts.

The system supports:

1. One integrated 3D generating-unit visualization.
2. Separate TGB and UGB component monitoring.
3. Shared generator power and unit speed measurements.
4. Linear Regression, LSTM, and GRU forecast selection.
5. Synchronized historical replay of November 2020 observations.
6. Next 10-minute vibration forecasting using previously generated model predictions.

The digital twin prototype does not use a live SCADA connection. Its current operation relies on historical sensor records and precomputed machine learning forecasts.

The next development stage integrates these capabilities into an interactive Streamlit dashboard.